# Bronze Layer Ingestion - Provider Drug Data
## Healthcare Lakehouse Project

This notebook ingests the **Provider Drug dataset** (~3.5 GB) from S3 into the Bronze layer using **Databricks Auto Loader**.

### Key Features:
- **Auto Loader (cloudFiles)**: Efficiently processes large volumes of new files
- **Schema Evolution**: Handles new columns with `addNewColumns` mode
- **Partitioning**: Data partitioned by read_timestamp for optimized queries
- **Metadata Tracking**: Captures file name and size for lineage
- **Delta Lake**: Writes to Delta format for ACID transactions

### Dataset Size: ~3.5 GB

### Step 1: Import Required Libraries

In [ ]:
# Databricks notebook source
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Step 2: Load Utility Variables
Runs the utilities notebook to access schema name variables.

In [ ]:
MAGIC
%run /Workspace/Users/pawanvirat32@gmail.com/healthcare_lakehouse_project/setup/utilities

### Step 3: Verify Schema Variables
Confirms that the schema variables are loaded correctly.

In [ ]:
print(bronze_schema, silver_schema, gold_schema)

### Step 4: Define Widget Parameters
Creates interactive widgets for runtime configuration:
- **catalog**: Unity Catalog name (default: healthcare_lakehouse)
- **data_source**: Source system identifier (default: cms)

In [ ]:
dbutils.widgets.text('catalog', 'healthcare_lakehouse', 'Catalog')
dbutils.widgets.text('data_source', 'cms', 'Data Source')

### Step 5: Get Widget Values
Retrieves the widget values and displays them for verification.

In [ ]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(f"catalog: {catalog}")
print(f"data_source: {data_source}")

### Step 6: Configure Auto Loader Stream for Drug Data
Sets up the Auto Loader configuration for the larger drug dataset:
- **cloudFiles.schemaEvolutionMode**: "addNewColumns" for automatic schema evolution
- **maxBytesPerTrigger**: 128MB chunks (smaller due to larger dataset size)
- **Schema Location**: Tracks schema changes over time

In [ ]:
df_provider = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_BASE}/by_provider_drug/schema")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.maxBytesPerTrigger", "128m")
    .load(f"{S3_BASE}/by_provider_drug/")
    .withColumn('read_timestamp', F.current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)

### Step 7: Write to Bronze Delta Table with Partitioning
Writes the streaming data to the Bronze layer Delta table:
- **Partition By**: read_timestamp for time-based partitioning
- **Checkpoint Location**: Tracks processing progress
- **mergeSchema**: Allows schema evolution
- **trigger.availableNow**: Processes all available data and stops

In [ ]:
(
    df_provider.writeStream
    .format("delta")
    .option("checkpointLocation", f"{CHECKPOINT_BASE}/by_provider_drug/checkpoint")
    .option("mergeSchema", "true")
    .partitionBy("read_timestamp")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"{TARGET_SCHEMA}.by_provider_drug")
)

### Step 8: Preview Data (Batch Mode)
Reads the data in batch mode to preview the ingested records.

In [ ]:
# Preview data by reading in batch mode
df_preview = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{S3_BASE}/by_provider_drug/")
)

In [ ]:
display(df_preview.limit(30))